<a href="https://colab.research.google.com/github/iamhardyyy/Next-Word-prediction-using-LSTM/blob/main/imdb_review_prediction_using_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [238]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense,Embedding,LSTM
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from nltk.corpus import stopwords

In [ ]:
df=pd.read_csv('/content/IMDB Dataset.csv')
df

In [ ]:
import nltk
sw=stopwords.words('english')
sw

In [221]:
def preprocess_dataset(df):
  x_data=df['review']
  y_data=df['sentiment']

  x_data=x_data.replace({'<.*?>':""},regex=True)
  x_data=x_data.replace({'[^A-z ]':""},regex=True)
  x_data=x_data.apply(lambda review:[w.lower() for w in review.split() if w.lower() not in sw])
  y_data=y_data.replace({'positive':1,'negative':0})

  return x_data,y_data

In [222]:
x_data,y_data=preprocess_dataset(df)

/tmp/ipykernel_32772/1865650606.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data=y_data.replace({'positive':1,'negative':0})


In [223]:
x_train,x_test,y_train,y_test=train_test_split(x_data,y_data,random_state=1,test_size=0.3)

In [224]:
x_train

,review
8950,"[note, add, comment, fear, black, hat, doesnt,..."
38421,"[one, worst, films, seen, date, pathetic, acti..."
19363,"[crossfire, one, films, forties, crying, remak..."
30157,"[film, begins, people, earth, discovering, roc..."
14294,"[pretty, good, episode, though, trapped, close..."
...,...
43723,"[largerthanlife, figures, wyatt, earp, bat, ma..."
32511,"[okay, havepenelope, keith, miss, herringbonet..."
5192,"[odd, willfully, skewed, biopic, dyan, thomas,..."
12172,"[basic, structure, story, beginning, middle, e..."


In [225]:
review_length=np.array([len(x) for x in x_train])

In [226]:
avg_length=int(np.mean(review_length))
avg_length

119

In [ ]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts(x_train)
tokenizer.word_index

In [228]:
x_train=tokenizer.texts_to_sequences(x_train)
x_test=tokenizer.texts_to_sequences(x_test)

In [229]:
x_train_padded=pad_sequences(x_train,maxlen=avg_length,truncating="post",padding='post')
x_test_padded=pad_sequences(x_test,maxlen=avg_length,truncating="post",padding='post')

In [230]:
voc_size=len(tokenizer.word_index)+1
voc_size

173328

In [231]:
model=Sequential()

In [232]:
model.add(Embedding(voc_size,32,input_length=avg_length))
model.add(LSTM(16))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [233]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [234]:
checkpoint=ModelCheckpoint('models/LSTM.h5',
                           monitor='accuracy',
                           save_best_only=True)

In [237]:
model.fit(x_train_padded,y_train,epochs=10,batch_size=64,callbacks=[checkpoint])

Epoch 1/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.5576 - loss: 0.6677

547/547 ━━━━━━━━━━━━━━━━━━━━ 58s 94ms/step - accuracy: 0.6356 - loss: 0.6295
Epoch 2/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.7114 - loss: 0.5945

547/547 ━━━━━━━━━━━━━━━━━━━━ 82s 95ms/step - accuracy: 0.7071 - loss: 0.6002
Epoch 3/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.7053 - loss: 0.5928

547/547 ━━━━━━━━━━━━━━━━━━━━ 83s 96ms/step - accuracy: 0.7100 - loss: 0.5835
Epoch 4/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 82s 95ms/step - accuracy: 0.6329 - loss: 0.6318
Epoch 5/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 81s 95ms/step - accuracy: 0.6178 - loss: 0.6365
Epoch 6/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 52s 95ms/step - accuracy: 0.6704 - loss: 0.5434
Epoch 7/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.8279 - loss: 0.4895

547/547 ━━━━━━━━━━━━━━━━━━━━ 83s 97ms/step - accuracy: 0.8435 - loss: 0.4342
Epoch 8/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 51s 93ms/step - accuracy: 0.7943 - loss: 0.4446
Epoch 9/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 53s 97ms/step - accuracy: 0.7560 - loss: 0.4844
Epoch 10/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 81s 95ms/step - accuracy: 0.7933 - loss: 0.4098


In [240]:
model.predict(x_test_padded)

469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step


array([[0.01298822],
       [0.218034  ],
       [0.00958122],
       ...,
       [0.21704394],
       [0.21492387],
       [0.23289423]], dtype=float32)

In [241]:
y_pred=model.predict(x_test_padded)>0.5
y_pred

469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step


array([[False],
       [False],
       [False],
       ...,
       [False],
       [False],
       [False]])

In [242]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.81      0.85      0.83      7521
           1       0.84      0.80      0.82      7479

    accuracy                           0.82     15000
   macro avg       0.83      0.82      0.82     15000
weighted avg       0.83      0.82      0.82     15000



In [243]:
model1=load_model('/content/models/LSTM.h5')

In [244]:
y_pred1=model1.predict(x_test_padded)

469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step


In [247]:
y_pred_new=y_pred1>0.5
y_pred_new

array([[False],
       [False],
       [False],
       ...,
       [False],
       [False],
       [False]])

In [248]:
print(classification_report(y_test,y_pred_new))

              precision    recall  f1-score   support

           0       0.81      0.82      0.82      7521
           1       0.82      0.81      0.81      7479

    accuracy                           0.81     15000
   macro avg       0.81      0.81      0.81     15000
weighted avg       0.81      0.81      0.81     15000



In [253]:
import re

In [278]:
def prediction(review):
  review=re.sub(r'[^A-z ]','',review)
  filtered=[w.lower() for w in review.split() if w.lower() not in sw]
  filtered=' '.join(filtered)
  t1=tokenizer.texts_to_sequences([filtered])
  padded=pad_sequences(t1,maxlen=avg_length,padding='post')
  return model1.predict(padded)

In [280]:
review=input('enter the review')
pred=prediction(review)
print(pred)
print('positive' if pred>0.5 else "negative")

enter the reviewit is a great movie, very good
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[[0.8326974]]
positive
